# 01 — Konsep Dasar Apache Spark (5 JP)

**Pelatihan Pengolahan Data dengan Spark — TA 2026**

Data latihan bersifat sintetis, bukan data wajib pajak sebenarnya.


## Objectives and timing
Theory 2 JP, practice 3 JP. Learn driver/executor, lazy evaluation, DAG, transformations vs actions; observe jobs and stages in Spark UI (`localhost:4040`).

**Instructor:** draw driver → master → workers → executors on whiteboard, then run each cell.


In [6]:
import os
from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder
    .appName("SparkTraining-Notebook")
    .master(os.getenv("SPARK_MASTER", "spark://spark-master:7077"))
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "4050")
    .config("spark.blockManager.port", "4051")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
print("Spark:", spark.version, "master:", spark.sparkContext.master)


Spark: 3.5.7 master: spark://spark-master:7077


### Transformations vs actions
A transformation creates a *plan*. An action triggers execution. Compare Spark UI before and after `count()`.


In [2]:
numbers = spark.range(0, 100_000, 1, numPartitions=4)
transformed = numbers.withColumn("mod10", F.col("id") % 10).filter(F.col("mod10") == 0)
print("Partitions:", transformed.rdd.getNumPartitions())
transformed.explain(mode="formatted")
print("Action count:", transformed.count())
transformed.groupBy("mod10").count().show()


Partitions: 4
== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 100000, step=1, splits=Some(4))

(2) Filter [codegen id : 1]
Input [1]: [id#0L]
Condition : ((id#0L % 10) = 0)

(3) Project [codegen id : 1]
Output [2]: [id#0L, (id#0L % 10) AS mod10#2L]
Input [1]: [id#0L]




Action count: 10000
+-----+-----+
|mod10|count|
+-----+-----+
|    0|10000|
+-----+-----+



In [7]:
print("Spark application ID:", spark.sparkContext.applicationId)
print("Web UI: http://localhost:4040 while this session is running")


Spark application ID: app-20261109061259-0000
Web UI: http://localhost:4040 while this session is running


26/11/09 06:20:15 ERROR Inbox: Ignoring error
java.util.concurrent.RejectedExecutionException: Task java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask@66c9c581[Not completed, task = java.util.concurrent.Executors$RunnableAdapter@74e5bcdf[Wrapped task = org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend$StandaloneDriverEndpoint$$anon$2@7e23e34d]] rejected from java.util.concurrent.ScheduledThreadPoolExecutor@79b3fd00[Terminated, pool size = 0, active threads = 0, queued tasks = 0, completed tasks = 0]
	at java.base/java.util.concurrent.ThreadPoolExecutor$AbortPolicy.rejectedExecution(Unknown Source)
	at java.base/java.util.concurrent.ThreadPoolExecutor.reject(Unknown Source)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor.delayedExecute(Unknown Source)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor.schedule(Unknown Source)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend$StandaloneDriverEndpoint.$anonfun$onDisc

### Lab (3 JP)
1. Explain which lines are transformations and actions.
2. Change input partitions to 1 and 8; compare tasks in Spark UI.
3. Draw the execution path and identify where shuffle occurs.
4. Explain why `explain()` itself does not execute the distributed query.

**Deliverable:** architecture diagram + one screenshot of DAG/stages.
